In [3]:
import os, base64, html, re, json                         
from typing import List, Optional                         
import pandas as pd                                      
from bs4 import BeautifulSoup                            
from google.oauth2.credentials import Credentials        
from google_auth_oauthlib.flow import InstalledAppFlow    
from googleapiclient.discovery import build               
from googleapiclient.errors import HttpError              
import torch                                              
from transformers import AutoTokenizer, AutoModelForSequenceClassification  
import matplotlib.pyplot as plt                           
from tqdm import tqdm                                     

# Configuration
SCOPES = ['https://www.googleapis.com/auth/gmail.readonly']   # Set Gmail scope to read-only
CREDENTIALS_PATH = 'credentials.json'                         # Set path to OAuth client credentials file
TOKEN_PATH = 'token.json'                                     # Set path to cached OAuth token file
# Try different folders, uncomment to check the folder
QUERY = 'in:inbox newer_than:7d'                            
#QUERY = 'in:spam'                                           
#QUERY = 'in:sent newer_than:30d'                            
#QUERY = 'in:inbox newer_than:7d -category:promotions -category:social'  
MAX_RESULTS = 50            # Set maximum number of messages to fetch

MODEL_DIR = '../models/saved_model'     # Local path to saved BERT model
LABELS = {0: "Ham", 1: "Phishing"}      # Map class indices to class names
THRESH_PHISH = 0.80                     # Set upper threshold for phishing classification
THRESH_HAM = 0.20                       # Set lower threshold for ham classification
MAX_BODY_CHARS = 3000                   # Set maximum number of body characters to keep
EXPORT_CSV = True                       # Enable or disable CSV export
EXPORT_PATH = '../analysis/gmail_prediction/gmail_predictions.csv'    # Set output CSV file path
# ==========================================================


# Preprocessing the emails
def normalize_punct_spacing_for_urls(s):                     
    s = str(s)                                               # Convert input to string
    s = re.sub(r'(?i)([a-z0-9])((?:https?|ftp|file))', r'\1 \2', s)  # Separate trailing text from protocol keyword
    s = re.sub(r'\s*\.\s*', '.', s)                          # Remove spaces around periods
    s = re.sub(r'\s*/\s*', '/', s)                           # Remove spaces around slashes
    s = re.sub(r'\s*:\s*', ':', s)                           # Remove spaces around colons
    s = re.sub(r'\b(?:h\s*t\s*t\s*p(?:s)?)\b',               # Match spaced-out http/https
               lambda m: m.group(0).replace(' ', ''),        # Collapse spaces inside protocol
               s, flags=re.IGNORECASE)
    s = re.sub(r'\b(?:f\s*t\s*p)\b',                         # Match spaced-out ftp
               lambda m: m.group(0).replace(' ', ''),        # Collapse spaces inside ftp
               s, flags=re.IGNORECASE)
    s = re.sub(r'(?i)\b(https?|ftp|file)\s*[\.:;]\s*/\s*/',  # Match broken protocol separators like https.// or http;//
               r'\1://',                                     # Replace with proper ://
               s)
    return s                                                 # Return normalized string

def strip_html(html_text: str) -> str:                  # Define helper to strip HTML tags and get text
    soup = BeautifulSoup(html_text, 'html.parser')      # Parse HTML content into BeautifulSoup object
    for tag in soup(['script', 'style']):               # Iterate over script and style tags
        tag.decompose()                                 # Remove script and style tags from tree
    text = soup.get_text(separator=' ', strip=True)     # Extract visible text with spaces as separators
    return ' '.join(text.split())                       # Collapse multiple spaces and return text

def clean_text_for_bert(text: str) -> str:         # Define cleaner for BERT input
    if text is None:                               # Check if input text is None
        return ""                                  # Return empty string if no text
    text = html.unescape(str(text))                # Decode HTML entities to plain text
    text = normalize_punct_spacing_for_urls(text)  # Normalize URL punctuation and spacing
    return re.sub(r'\s+', ' ', text).strip()       # Collapse whitespace and trim edges

# Gmail helpers
def gmail_service_popup():                                   
    creds = None                                             
    if os.path.exists(TOKEN_PATH):                              # Check if cached token file exists
        creds = Credentials.from_authorized_user_file(TOKEN_PATH, SCOPES)  # Load cached credentials
    if not creds or not creds.valid:                            # Check if credentials are missing or invalid
        if creds and getattr(creds, "refresh_token", None):     # Check if refresh token is available
            from google.auth.transport.requests import Request  # Import Request for token refresh
            creds.refresh(Request())                            # Refresh credentials using refresh token
        else:                                                   # Case with no valid credentials
            flow = InstalledAppFlow.from_client_secrets_file(   # Create OAuth flow from client secrets
                CREDENTIALS_PATH, SCOPES
            )
            try:                                             
                creds = flow.run_local_server(port=0, open_browser=True)  # Launch browser for OAuth consent
            except Exception:                                # Handle failure to open browser
                print("Could not launch browser. Please open the displayed URL manually.")  
                creds = flow.run_console()                   # Run console-based OAuth flow
        with open(TOKEN_PATH, 'w') as f:                     # Open token file for writing
            f.write(creds.to_json())                         # Save credentials to token file
    return build('gmail', 'v1', credentials=creds)           # Build and return Gmail API service object

def base64url_decode(data: str) -> bytes:                    # Decode base64url-encoded data
    return base64.urlsafe_b64decode(data + '==')             # Decode with padding adjustment

def get_header(headers: List[dict], name: str) -> Optional[str]:  # Define helper to fetch header by name
    for h in headers:                   # Iterate over header list
        if h.get('name') == name:       # Check if header name matches target
            return h.get('value')       # Return header value if found
    return None                         # Return None if header not found

def extract_plain_text_from_payload(payload: dict) -> str:   # Extract plain text from Gmail payload
    texts = []                                               
    def handle_part(p):                     # Define nested function to handle one MIME part
        mime = p.get('mimeType', '')        # Get MIME type of part
        body = p.get('body', {})            # Get body dictionary for part
        data = body.get('data')             # Get base64 data from body if present
        if 'parts' in p:             # Check if part contains subparts (multipart)
            for sub in p['parts']:          # Iterate over subparts
                handle_part(sub)            # Recursively handle each subpart
        else:                        # Handle non-multipart part
            if data:                                         # Check if base64 data exists
                content = base64url_decode(data).decode('utf-8', errors='ignore')  # Decode content to UTF-8
                if mime.startswith('text/plain'):            # Check if plain text content
                    texts.append(content)                    # Append plain text to list
                elif mime.startswith('text/html'):           # Check if HTML content
                    texts.append(strip_html(content))        # Convert HTML to text and append
    if 'parts' in payload:                         # Check if root payload is multipart
        for part in payload['parts']:              # Iterate over root parts
            handle_part(part)                      # Process each part with handler
    else:                                          # Handle non-multipart root payload
        body_root = payload.get('body', {})        # Get root body dictionary
        data_root = body_root.get('data')          # Get root base64 data
        mime_root = payload.get('mimeType', '')    # Get root MIME type
        if data_root:                              # Check if root data exists
            content = base64url_decode(data_root).decode('utf-8', errors='ignore')  # Decode root content
            if mime_root.startswith('text/plain'):           # Check if root is plain text
                texts.append(content)                        # Append plain text to list
            elif mime_root.startswith('text/html'):          # Check if root is HTML
                texts.append(strip_html(content))            # Convert HTML to text and append
    return "\n\n".join([t for t in texts if t]).strip()      # Join non-empty parts and trim whitespace

# Fetch emails & clean 
service = gmail_service_popup()     # Create authenticated Gmail service

try:
    resp = service.users().messages().list(         # Call Gmail API to list messages
        userId='me', q=QUERY, maxResults=MAX_RESULTS
    ).execute()
except HttpError as e:                                       
    print("Gmail API error:", e)     
    raise                 # Reraise exception for visibility

messages = resp.get('messages', []) or []                    # Extract message list or use empty list
print(f"Found {len(messages)} messages for query: {QUERY}")  # Print number of messages found

rows = []                                                   
for m in messages:                                           # Iterate over each message stub
    msg = service.users().messages().get(                    # Fetch full message content
        userId='me', id=m['id'], format='full'
    ).execute()
    payload = msg.get('payload', {})                         # Get payload from message
    headers = payload.get('headers', [])                     # Get header list from payload
    subj = get_header(headers, 'Subject') or ''              # Extract Subject header
    frm  = get_header(headers, 'From') or ''                 # Extract From header
    body = extract_plain_text_from_payload(payload)          # Extract plain text body from payload
    body = body[:MAX_BODY_CHARS] if MAX_BODY_CHARS else body # Truncate body length if reached limit (3000)
    rows.append({                                            # Append record to row list
        "id": m["id"],                                       # Store message ID
        "from": frm,                                         # Store sender address
        "subject": clean_text_for_bert(subj),                # Store subject text
        "body": clean_text_for_bert(body),                   # Store body text
    })

df = pd.DataFrame(rows)                      # Create DataFrame from collected rows
print(f"Fetched {len(df)} messages.")        # Print number of messages fetched
if len(df) == 0:                             # Check if no messages are available
    display(df)                              # Display empty DataFrame for inspection
    raise SystemExit()                      


# Load the pretrained model
device = 'cuda' if torch.cuda.is_available() else 'cpu'      # Select GPU if available, else CPU

try:
    tokenizer = AutoTokenizer.from_pretrained(               # Load tokenizer from local model directory
        MODEL_DIR, local_files_only=True
    )
    model = AutoModelForSequenceClassification.from_pretrained(  # Load model from local model directory
        MODEL_DIR, local_files_only=True
    ).to(device).eval()                                      # Move model to device and set eval mode
except Exception as e:                                       
    tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)     # Load tokenizer with default
    model = AutoModelForSequenceClassification.from_pretrained(  # Load model with default loader
        MODEL_DIR
    ).to(device).eval()                                      # Move model to device and set eval mode


# Inference / Prediction process
@torch.inference_mode()                                      # Disable gradient tracking for inference
def score_batch(texts):                                      # Define helper to score a list of texts
    outs = []                                                # Initialize list for phishing probabilities
    for t in tqdm(texts, desc="Scoring emails"):             # Iterate over texts with progress bar
        enc = tokenizer(                                     # Tokenize single email text
            t,
            truncation=True,
            max_length=512,
            padding=False,
            return_tensors='pt'
        )
        enc = {k: v.to(device) for k, v in enc.items()}      # Move tokenized inputs to selected device
        logits = model(**enc).logits[0]                      # Run model forward pass and take first output row
        probs = torch.softmax(logits, dim=-1).cpu().tolist() # Convert logits to probabilities
        p_phish = probs[1]                                   # Extract probability for phishing class 
        outs.append(p_phish)                                 # Append phishing probability to output list
    return outs                                              # Return list of phishing probabilities

df["p_phish"] = score_batch(df["body"].tolist())             # Score all email bodies and store

def classify(p):                                             # Define helper to map probability to label
    if p >= THRESH_PHISH: return "Phishing"                  # Label as Phishing if above upper threshold
    if p <= THRESH_HAM:   return "Ham"                       # Label as Ham if below lower threshold
    return "Uncertain"                                       # Label as Uncertain if probability is in the middle

df["pred"] = df["p_phish"].apply(classify)                   # Apply classifier to probability

# Display predictions
view_cols = ["from", "subject", "p_phish", "pred"]           # Display column "From", "Subject", "Probability of the prediction", "Prediction"
df_view = (                                                   # Build sorted view DataFrame
    df[view_cols]                                            # Slice columns of interest
      .sort_values("p_phish", ascending=False)               # Sort rows by phishing probability
      .reset_index(drop=True)                                # Reset index after sorting
)
display(df_view)                                             # Show ranked emails with probabilities and labels

# Export predictions
if EXPORT_CSV:                                               # Check if CSV export is enabled
    os.makedirs(os.path.dirname(EXPORT_PATH), exist_ok=True) # Create output folder if missing
    df.to_csv(EXPORT_PATH, index=False, encoding="utf-8")    # Save full DataFrame to CSV file
    print(f"Saved predictions to {EXPORT_PATH}")             # Print confirmation of save path



Found 50 messages for query: in:inbox newer_than:7d
Fetched 50 messages.


Scoring emails: 100%|███████████████████████████████████████████████████████████████████| 50/50 [00:00<00:00, 61.48it/s]


,from,subject,p_phish,pred
0,Kijiji Canada <no-reply@shop.kijiji.ca>,Winter tires or a whole new ride? We’ve got yo...,0.999901,Phishing
1,Papa Johns <offers@promotions.papajohns.com>,The Butter Chicken pizza you crave,0.999894,Phishing
2,"""Scene+"" <news@news.sceneplus.ca>","Quan, check out this week’s grocery offers! 🛒✨",0.999783,Phishing
3,My Grocery Offers by Sobeys <sobeysoffers@em.s...,"Quan, your holiday prep starts with these offers!",0.999730,Phishing
4,Papa Johns <offers@promotions.papajohns.com>,Two pizzas means twice the flavour,0.999715,Phishing
5,Value Village <marketing@goto.valuevillage.com>,Fundraise for your nonprofit.Every dollar counts.,0.999702,Phishing
6,Fido <Fido@e.fido.ca>,"Hi Quan, welcome to Fido! 🐶",0.999647,Phishing
7,Fido <Fido@e.fido.ca>,"Hi Quan, learn about your first bill before it...",0.999625,Phishing
8,The Keg <reply@e.kegrestaurants.com>,Our Kansas City Striploin is back,0.999450,Phishing
9,Google <no-reply@accounts.google.com>,Your Google Account was recovered successfully,0.998606,Phishing


Saved predictions to ../analysis/gmail_prediction/gmail_predictions.csv
